# int

Integers power counters, IDs, retry budgets, timestamps, and capacity boundaries. If integer behavior is fuzzy, production bugs look like business logic bugs instead of math bugs.


## 1. Problem First

Why does this matter in real systems?

- Rate limit counters depend on exact arithmetic.
- Retry budgets are often integer-based.
- Off-by-one mistakes silently drop work.


In [ ]:
retry_limit = 3
attempts = 0

while attempts < retry_limit:
    attempts += 1
    print(f"attempt={attempts}")


## 2. Minimal Concept

Python `int` represents whole numbers.

- Positive: `10`
- Zero: `0`
- Negative: `-7`
- Operators: `+`, `-`, `*`, `//`, `%`, `**`


## 3. Mental Model

How Python actually behaves

Python integers are immutable objects. Reassigning a variable does not mutate a box in memory; it binds the name to a different integer object. Python integers also support arbitrary precision, so they do not overflow like fixed-width machine integers.


In [ ]:
a = 10
print(id(a), a)
a += 1
print(id(a), a)

huge = 10 ** 50
print(huge)


## 4. Internal Mechanics

Small integers are often cached by CPython, which is why some equal values may share identity. That is an optimization, not a correctness contract.


In [ ]:
import dis

x = 256
y = 256
print(x is y)

def increment(value):
    value += 1
    return value

dis.dis(increment)


## 5. Edge Cases

Where it breaks

- `//` floors toward negative infinity.
- `%` with negative numbers surprises people.
- `/` returns `float`, not `int`.


In [ ]:
print(-3 // 2)
print(-3 % 2)
print(5 / 2, type(5 / 2))
print(5 // 2, type(5 // 2))


## 6. Performance Thinking

- Small integer arithmetic is effectively O(1).
- Large integers cost more as digit length grows.
- `int` avoids floating-point rounding in exact-count problems.
- Arbitrary precision trades memory and speed for correctness.


## 7. Real Use Case

Counting error events in a log pipeline is an exact integer problem.


In [ ]:
events = ["INFO", "ERROR", "ERROR", "WARN", "ERROR"]
error_count = 0

for level in events:
    if level == "ERROR":
        error_count += 1

print(error_count)


## 8. Anti-Pattern

What beginners do wrong

- Use `/` and accidentally introduce floats.
- Assume negative division behaves the same across languages.
- Use `is` for numeric value comparison.


In [ ]:
processed = 5 / 2
print(processed, type(processed))

safe_processed = 5 // 2
print(safe_processed, type(safe_processed))


## 9. Interview Signals

Questions FAANG asks

- Why do Python integers not overflow like Java `int`?
- What is the difference between `/` and `//`?
- Why does `-3 // 2` equal `-2`?
- When should you prefer `int` over `float`?


## 10. Exercise (Non-trivial)

Design a rate limiter with fixed-size windows. Make rollover correct, prevent off-by-one bugs, and keep the logic easy to audit under incident pressure.


In [ ]:
def can_send(current_count, limit):
    return current_count < limit

# Task:
# 1. Add window reset logic.
# 2. Prevent off-by-one mistakes.
# 3. Explain where integer arithmetic keeps this exact.
